# Etapa 5 - Análise de precificação e variação de margem

Notebook executável da Etapa 5. A lógica canônica fica em `etapa5_precificacao_margem.py`; este notebook reexecuta o script e apresenta os principais arquivos gerados.

<!-- glossario-comercial -->
## 📖 Glossário rápido (para leitura não técnica)

Esta seção traduz os termos técnicos do notebook para linguagem de negócio. Quem é de áreas como
comercial, marketing ou compras pode ler os resultados sem precisar do detalhe técnico.

| Termo | O que significa, em linguagem comercial |
|---|---|
| **SKU** | Código único de um produto. Cada item distinto do catálogo é um SKU (cor/tamanho/voltagem diferentes contam como SKUs diferentes). |
| **Unidade de armazenagem** | A unidade padrão de contagem do estoque (p.ex. a caixa). Convertemos preço e custo para ela para não comparar "preço da caixa" com "custo da unidade" como se fossem iguais. |
| **Preço praticado** | O preço médio que de fato saiu no caixa, por unidade de armazenagem (`receita ÷ quantidade`). É o preço real de venda, não o de tabela. |
| **Custo médio (CMV)** | Quanto custou, em média, repor uma unidade — a média ponderada do preço de compra no período. Só existe para itens que tiveram compra registrada. |
| **Margem bruta (R$)** | Quanto sobra em reais por unidade depois de pagar o custo: preço praticado − custo médio. |
| **Margem bruta (%)** | Quanto sobra em proporção do preço: margem R$ ÷ preço praticado. Margem 50% = metade do preço é lucro bruto. |
| **Markup** | Quantas vezes o preço cobre o custo (preço ÷ custo). Markup 2× = preço é o dobro do custo. |
| **Preço de lista** | O preço de tabela cadastrado para o produto em cada loja e embalagem — o preço "cheio", antes de desconto. |
| **Desconto efetivo (%)** | Quanto, em média, o preço praticado ficou abaixo da tabela: `(lista − praticado) ÷ lista`. Sempre comparado dentro da mesma embalagem. |
| **Dispersão de preço** | O quanto o preço do mesmo produto varia entre lojas. Medido por embalagem, para não confundir "caixa" com "unidade". |
| **Repricing** | Revisão de preço de itens com margem baixa/negativa, desconto fora do padrão ou preço fora da faixa da rede. |
| **Curva ABC** | Regra de Pareto aplicada à receita: a **classe A** são os poucos produtos que somam ~80% do faturamento; B e C têm peso decrescente. |
| **Cobertura de custo** | Quanto da receita está coberta por itens com custo conhecido. Aqui é ~16%: a margem realizada só vale para esse subconjunto. |
| **Rede física × atacado/B2B (Loja 93)** | A Loja 93 vende em grande volume para outras empresas (atacado), com preços e margens diferentes do varejo. Por isso é mostrada à parte. |
| **Proxy** | Uma medida aproximada usada quando o dado ideal não existe (ex.: preço de lista sem vigência de promoção para estimar desconto). |
| **Parquet** | Formato de arquivo compactado e rápido para grandes volumes de dados, usado internamente para acelerar o processamento. |


## Metodologia resumida

- **Unidades.** Tudo é comparado na unidade de armazenagem. Preço praticado = `RECEITA / QTD_ARMAZENAGEM`;
  custo médio = `PRECO_UNIT_UNIDADE_COMPRA / CONVERSAO_COMPRA_ARMAZENAGEM` (ponderado pela quantidade comprada).
  O desconto efetivo compara preço praticado e preço de lista **dentro da mesma embalagem**.
- **Cobertura de custo.** Margem realizada só existe para os SKUs com compra de preço válido (~261 SKUs, ~16% da
  receita) — análogo ao achado "88% vendem sem compra registrada" das Etapas 1/2. Os demais ficam sem margem por
  ausência de dado, nunca por imputação.
- **Universos.** Toda saída agregada traz `UNIVERSO` com `REDE_COMPLETA` e `REDE_FISICA_SEM_LOJA93`. A Loja 93
  (atacado/B2B) entra na rede completa mas é segregada; o custo de cada universo usa apenas as compras das lojas do universo.
- **Reconciliação.** Receita por universo, categoria e loja reconcilia com os outputs auditáveis da Etapa 3.

> **Como ler a margem:** ela é **realizada e histórica** (média do período), sobre o subconjunto com custo. Não é
> margem corrente nem prospectiva (sem camadas de custo/PEPS nem custo de reposição). Margens negativas (preço < custo)
> são **sinalizadas, não silenciadas**. A dispersão de preço e o desconto efetivo corrigem leituras ingênuas que
> misturavam embalagem (caixa × unidade) e atacado × varejo.


In [1]:
import runpy
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs' / 'etapa5'

runpy.run_path(str(ROOT / 'notebooks' / 'etapa5_precificacao_margem.py'), run_name='__main__')

Carregando bases...
Calculando custo medio por SKU (rede completa e rede fisica)...


Agregando margem por categoria, loja e total...


Calculando margem por SKU, precificacao, dispersao e candidatos...


Autoaudit (antes/depois)...


Validando reconciliacoes...
Salvando arquivos auditaveis...



--- Destaques Etapa 5 ---
SKUs com custo: 261 de 2729 (16.4% da receita)
Margem % rede completa: 47.6% | rede fisica: 49.8%
Margem % Loja 93/B2B: 43.9%
Markup rede fisica: 1.99x
Candidatos a repricing (rede fisica): 3260
Validacoes OK: 21/21

[OK] Arquivos salvos em outputs/etapa5/
  outputs\etapa5\autoaudit_etapa5.csv
  outputs\etapa5\candidatos_repricing.csv
  outputs\etapa5\candidatos_repricing_impacto.csv
  outputs\etapa5\dispersao_preco_lojas.csv
  outputs\etapa5\documentacao_tecnica_etapa5.md
  outputs\etapa5\margem_categorias_n1.csv
  outputs\etapa5\margem_categorias_n2.csv
  outputs\etapa5\margem_categorias_n3.csv
  outputs\etapa5\margem_lojas.csv
  outputs\etapa5\margem_produtos.csv
  outputs\etapa5\margem_total_universo.csv
  outputs\etapa5\precificacao_desconto.csv
  outputs\etapa5\recomendacoes_melhoria.csv
  outputs\etapa5\resumo_etapa5.md
  outputs\etapa5\validacoes_etapa5.csv


{'__name__': '__main__',
 '__doc__': '\netapa5_precificacao_margem.py\n=============================\nEtapa 5 - Analise de precificacao e variacao de margem\n\nObjetivos\n---------\n1. Calcular margem bruta realizada (R$ e %), markup e custo medio por SKU,\n   categoria e loja, usando preco praticado e custo na MESMA unidade de\n   armazenagem.\n2. Comparar preco praticado com o preco de lista (`dim_precos`) e medir o\n   desconto efetivo, sempre dentro da MESMA embalagem.\n3. Medir a dispersao de preco do mesmo SKU entre lojas, separando por\n   embalagem para nao misturar caixa com unidade.\n4. Priorizar candidatos a repricing (margem baixa/negativa em alta receita,\n   desconto efetivo acima da media e preco fora da faixa da rede).\n5. Separar rede completa, rede fisica sem Loja 93 e o canal atacado/B2B, e\n   documentar de forma transparente a cobertura de custo e as limitacoes.\n\nPremissas e decisoes metodologicas\n-----------------------------------\n- Unidades. Tudo e normaliza

## Arquivos gerados

In [2]:
arquivos = pd.DataFrame(
    {
        'arquivo': [p.name for p in sorted(OUT.glob('*'))],
        'tamanho_kb': [round(p.stat().st_size / 1024, 1) for p in sorted(OUT.glob('*'))],
    }
)
display(arquivos)

,arquivo,tamanho_kb
0,autoaudit_etapa5.csv,2.1
1,candidatos_repricing.csv,2619.7
2,candidatos_repricing_impacto.csv,2619.7
3,dispersao_preco_lojas.csv,936.4
4,documentacao_tecnica_etapa5.md,4.8
5,margem_categorias_n1.csv,9.4
6,margem_categorias_n2.csv,46.8
7,margem_categorias_n3.csv,174.8
8,margem_lojas.csv,5.0
9,margem_produtos.csv,132.4


## Validações

Todas as reconciliações devem ter status `OK`; qualquer `FALHA` bloqueia conclusões.

In [3]:
validacoes = pd.read_csv(OUT / 'validacoes_etapa5.csv', encoding='utf-8-sig')
print(f"Validações OK: {(validacoes['STATUS'] == 'OK').sum()}/{len(validacoes)}")
display(validacoes)

Validações OK: 21/21


,VALIDACAO,OBSERVADO,ESPERADO,DIFERENCA,STATUS
0,Receita rede completa vs Etapa 3,4.825157e+08,4.825157e+08,0.0,OK
1,Receita rede fisica vs Etapa 3,3.292348e+08,3.292348e+08,0.0,OK
2,Receita Loja 93/B2B vs Etapa 3,1.532810e+08,1.532810e+08,0.0,OK
3,REDE_COMPLETA = REDE_FISICA + Loja 93,4.825157e+08,4.825157e+08,-0.0,OK
4,Universo Loja 93/B2B gerado nas saidas agregadas,1.000000e+00,1.000000e+00,0.0,OK
5,Categoria N1: receita vs Etapa 3 (max abs),0.000000e+00,0.000000e+00,0.0,OK
6,Loja: receita vs Etapa 3 (max abs),0.000000e+00,0.000000e+00,0.0,OK
7,Margem por SKU so com custo presente,1.000000e+00,1.000000e+00,0.0,OK
8,SKUs com margem = SKUs com custo (rede completa),2.610000e+02,2.610000e+02,0.0,OK
9,Margem % dentro da faixa sanitaria,1.000000e+00,1.000000e+00,0.0,OK


## Margem por categoria (rede física)

Margem agregada e ponderada apenas sobre o subconjunto com custo. `COBERTURA_CUSTO_RECEITA_PCT` mostra quanto da receita da categoria tem custo conhecido — margens de categorias com cobertura muito baixa não são representativas.

In [4]:
cat = pd.read_csv(OUT / 'margem_categorias_n1.csv', encoding='utf-8-sig')
cols = ['NIVEL_1', 'RECEITA_TOTAL', 'RECEITA_COM_CUSTO', 'COBERTURA_CUSTO_RECEITA_PCT',
        'MARGEM_BRUTA_PCT', 'MARKUP_PONDERADO', 'SKUS', 'SKUS_COM_CUSTO']
display(cat[cat['UNIVERSO'] == 'REDE_FISICA_SEM_LOJA93'][cols].head(15).round(2))

,NIVEL_1,RECEITA_TOTAL,RECEITA_COM_CUSTO,COBERTURA_CUSTO_RECEITA_PCT,MARGEM_BRUTA_PCT,MARKUP_PONDERADO,SKUS,SKUS_COM_CUSTO
28,I - ILUMINACAO,25321216.15,15914310.02,62.85,51.31,2.05,213,76
29,C - PISOS E REVESTIMENTOS,54643661.47,11532347.37,21.10,56.13,2.28,325,34
30,D - ELETROS,81442197.07,10842532.87,13.31,42.83,1.75,286,35
31,E - MATERIAIS ELETRICOS,11332398.32,6763846.32,59.69,53.31,2.14,111,47
32,O - MOVEIS,13649986.05,2819442.10,20.66,46.41,1.87,193,13
33,U - METAIS E ACESSORIOS,20361396.72,628538.35,3.09,40.58,1.68,142,1
34,B - UTILIDADES DOMESTICAS,22513813.98,597346.63,2.65,-15.43,0.87,529,12
35,"L - INFANTIL, LAZER E PETS",827989.30,391242.86,47.25,53.74,2.16,27,11
36,"K - CAMA, MESA E BANHO",23212282.18,195688.89,0.84,69.57,3.29,358,12
37,"G - PORTAS, JANELAS E FECHADURAS",5242924.64,165941.89,3.17,56.66,2.31,39,7


## Margem por loja

In [5]:
lojas = pd.read_csv(OUT / 'margem_lojas.csv', encoding='utf-8-sig')
cols = ['UNIVERSO', 'COD_EMPRESA', 'CD_CIDADE', 'CD_ESTADO', 'RECEITA_TOTAL',
        'COBERTURA_CUSTO_RECEITA_PCT', 'MARGEM_BRUTA_PCT', 'MARKUP_PONDERADO']
display(lojas[cols].round(2))

,UNIVERSO,COD_EMPRESA,CD_CIDADE,CD_ESTADO,RECEITA_TOTAL,COBERTURA_CUSTO_RECEITA_PCT,MARGEM_BRUTA_PCT,MARKUP_PONDERADO
0,LOJA_93_ATACADO_B2B,93,ALHANDRA,PB,1.532810e+08,12.83,43.91,1.78
1,REDE_COMPLETA,93,ALHANDRA,PB,1.532810e+08,17.77,44.13,1.79
2,REDE_COMPLETA,2,RECIFE,PE,5.337709e+07,16.96,49.90,2.00
3,REDE_COMPLETA,3,SALVADOR,BA,6.259999e+07,13.69,52.69,2.11
4,REDE_COMPLETA,4,RECIFE,PE,4.330823e+07,14.35,47.39,1.90
5,REDE_COMPLETA,6,JOAO PESSOA,PB,3.481680e+07,15.68,49.04,1.96
6,REDE_COMPLETA,92,CABO DE SANTO AGOSTINHO,PE,2.856488e+07,18.05,43.73,1.78
7,REDE_COMPLETA,8,CARUARU,PE,2.134132e+07,19.00,49.53,1.98
8,REDE_COMPLETA,5,ARACAJU,SE,2.352973e+07,15.35,50.67,2.03
9,REDE_COMPLETA,9,SALVADOR,BA,2.106606e+07,16.22,51.40,2.06


## Top e bottom SKUs por margem (alta receita)

Entre os itens de alta receita (curva A) com custo, os de maior e menor margem. As margens negativas (preço < custo) são sinalizadas, não removidas.

In [6]:
sku = pd.read_csv(OUT / 'margem_produtos.csv', encoding='utf-8-sig')
sku_a = sku[(sku['UNIVERSO'] == 'REDE_FISICA_SEM_LOJA93') & (sku['CURVA_ABC_RECEITA'] == 'A')]
cols = ['CODIGO', 'DESCRICAO', 'NIVEL_1', 'RECEITA', 'PRECO_PRATICADO_ARM',
        'CUSTO_MEDIO_ARM', 'MARGEM_BRUTA_PCT', 'MARKUP', 'FLAG_MARGEM_NEGATIVA']
print('Maiores margens (curva A):')
display(sku_a.sort_values('MARGEM_BRUTA_PCT', ascending=False)[cols].head(8).round(2))
print('Menores margens (curva A):')
display(sku_a.sort_values('MARGEM_BRUTA_PCT')[cols].head(8).round(2))

Maiores margens (curva A):


,CODIGO,DESCRICAO,NIVEL_1,RECEITA,PRECO_PRATICADO_ARM,CUSTO_MEDIO_ARM,MARGEM_BRUTA_PCT,MARKUP,FLAG_MARGEM_NEGATIVA
392,462811,DISJ.BIPO.C DIN 16A GII-C016A,E - MATERIAIS ELETRICOS,101580.34,50.44,15.18,69.90,3.32,0
328,326263,SUPORTE 4X2 3MOD. BR 850200DUALE UP,E - MATERIAIS ELETRICOS,261051.40,1.75,0.54,69.46,3.27,0
360,462809,DISJ.BIPO.C DIN 32A GII-C032A,E - MATERIAIS ELETRICOS,158976.40,49.63,15.18,69.41,3.27,0
343,483606,"CER.A 60X60 COLOSSEO EXT RT 2,12M2",C - PISOS E REVESTIMENTOS,204861.74,56.68,17.39,69.32,3.26,0
390,462808,DISJ.BIPO.C DIN 25A GII-C025A,E - MATERIAIS ELETRICOS,103152.93,49.43,15.18,69.29,3.26,0
378,462810,DISJ.BIPO.C DIN 40A GII-C040A,E - MATERIAIS ELETRICOS,125816.98,50.84,15.71,69.10,3.24,0
387,475211,BUFFET 1.59X0.45 2P ALDO AMD/OFF,O - MOVEIS,107767.17,810.28,251.55,68.95,3.22,0
385,429458,TOM.MOD 2P+T 20A LG03020 POP,E - MATERIAIS ELETRICOS,111378.51,9.61,3.06,68.22,3.15,0


Menores margens (curva A):


,CODIGO,DESCRICAO,NIVEL_1,RECEITA,PRECO_PRATICADO_ARM,CUSTO_MEDIO_ARM,MARGEM_BRUTA_PCT,MARKUP,FLAG_MARGEM_NEGATIVA
359,447868,PRATO RASO PORC.28CM PEIXES 80000,B - UTILIDADES DOMESTICAS,162292.33,17.17,26.24,-52.76,0.65,1
376,467615,AP.JANTAR PORC.20PC NATUR 990100,B - UTILIDADES DOMESTICAS,129854.93,333.82,408.57,-22.39,0.82,1
333,432628,"FRITAD.ELET 4,2L ULTRA TOUCH MF M220",D - ELETROS,246973.06,329.30,378.78,-15.03,0.87,1
299,477961,LUSTRE LED 3EM1 20W ACR-003D 6058,I - ILUMINACAO,602573.51,442.09,417.16,5.64,1.06,0
282,429455,TOM.CJ.1T 2P+T 10A LGX030 POP,E - MATERIAIS ELETRICOS,1299370.49,4.62,3.57,22.75,1.29,0
305,429459,INT.SPL CJ.1S 10A LGZ010 POP,E - MATERIAIS ELETRICOS,502701.85,4.58,3.07,32.89,1.49,0
382,478112,REFRIG. 2P 490L FF INV. IB7 BR M220,D - ELETROS,122813.00,4548.63,2924.39,35.71,1.56,0
288,470050,LUSTRE LED 35W NE LP-201D 5629,I - ILUMINACAO,898885.80,470.87,298.83,36.54,1.58,0


## Precificação vs. lista e desconto efetivo

Preço praticado comparado ao preço de lista **dentro da mesma embalagem**. Desconto negativo = preço praticado acima da lista (possível tabela desatualizada).

In [7]:
pd_desc = pd.read_csv(OUT / 'precificacao_desconto.csv', encoding='utf-8-sig')
fis = pd_desc[(pd_desc['UNIVERSO'] == 'REDE_FISICA_SEM_LOJA93') & pd_desc['DESCONTO_EFETIVO_PCT'].notna()]
import numpy as np
resumo_desc = pd.DataFrame({
    'metrica': ['combinações loja×produto×embalagem com lista', 'desconto efetivo médio (ponderado receita)',
                'desconto efetivo mediano', 'combinações acima da lista (desconto < 0)',
                'combinações com desconto > 30%'],
    'valor': [len(fis),
              round(np.average(fis['DESCONTO_EFETIVO_PCT'], weights=fis['RECEITA']), 2),
              round(fis['DESCONTO_EFETIVO_PCT'].median(), 2),
              int((fis['DESCONTO_EFETIVO_PCT'] < 0).sum()),
              int((fis['DESCONTO_EFETIVO_PCT'] > 30).sum())],
})
display(resumo_desc)

,metrica,valor
0,combinações loja×produto×embalagem com lista,21205.00
1,desconto efetivo médio (ponderado receita),17.39
2,desconto efetivo mediano,12.81
3,combinações acima da lista (desconto < 0),2079.00
4,combinações com desconto > 30%,2956.00


## Dispersão de preço entre lojas (por embalagem)

Coeficiente de variação do preço praticado do mesmo SKU entre lojas, **separado por embalagem** — corrige a leitura ingênua que misturava caixa com unidade.

In [8]:
disp = pd.read_csv(OUT / 'dispersao_preco_lojas.csv', encoding='utf-8-sig')
disp_f = disp[disp['UNIVERSO'] == 'REDE_FISICA_SEM_LOJA93']
print(f"SKUs×embalagem com >=2 lojas: {len(disp_f)} | com CV>30%: {int((disp_f['CV_PRECO']>0.30).sum())}")
cols = ['CODIGO', 'DESCRICAO', 'NIVEL_1', 'EMBALAGEM', 'LOJAS', 'PRECO_ARM_MEDIO',
        'PRECO_ARM_MIN', 'PRECO_ARM_MAX', 'CV_PRECO']
display(disp_f.sort_values('CV_PRECO', ascending=False)[cols].head(10).round(2))

SKUs×embalagem com >=2 lojas: 2742 | com CV>30%: 46


,CODIGO,DESCRICAO,NIVEL_1,EMBALAGEM,LOJAS,PRECO_ARM_MEDIO,PRECO_ARM_MIN,PRECO_ARM_MAX,CV_PRECO
2745,426271,"REVEST.A 30X90 WHITE HIGHW.MAT1,07M2",C - PISOS E REVESTIMENTOS,0,2,91.93,50.30,133.56,0.64
2746,432465,XICARA CAFE CER.75ML C/P RS 102441,B - UTILIDADES DOMESTICAS,0,3,15.85,8.13,26.76,0.61
2747,446697,BALCAO COZ. MDF CANTO FRJ/GRF 500341,O - MOVEIS,0,5,470.20,209.42,839.90,0.57
2748,372512,"PORCEL.A 90X90 ESM.MUNARI M.EX1,63M2",C - PISOS E REVESTIMENTOS,0,2,87.52,52.40,122.65,0.57
2749,489208,CADEIRA SALA LINHO MAD 3285 BG 2564,O - MOVEIS,0,2,333.70,209.90,457.51,0.52
2750,421455,"REVEST.A 35X59 RVE35420 2,05M2",C - PISOS E REVESTIMENTOS,0,2,32.81,20.90,44.72,0.51
2751,471329,XICARA CAFE PORC.75ML C/P 01298,B - UTILIDADES DOMESTICAS,0,3,13.31,8.80,20.90,0.50
2752,181513,BACIA P/CX.M.CARLO P808 EB95 EBANO,W - LOUCAS E PIAS,0,7,722.41,314.90,1151.40,0.49
2753,407150,PORTA-BOLO IX 32CM C/TP 1525/499,B - UTILIDADES DOMESTICAS,0,3,46.62,24.04,65.53,0.45
2754,261159,TORNEIRA ELET.MESA CR SENS.1188 M220,U - METAIS E ACESSORIOS,0,5,3248.64,1364.90,4756.40,0.43


## Candidatos a repricing

Ranking de oportunidades por nº de sinais (margem baixa/negativa, desconto alto, preço fora da faixa) e receita.

In [9]:
cand = pd.read_csv(OUT / 'candidatos_repricing.csv', encoding='utf-8-sig')
cand_f = cand[cand['UNIVERSO'] == 'REDE_FISICA_SEM_LOJA93']
print(f"Total candidatos (rede física): {len(cand_f)} | ALTA: {(cand_f['FAIXA_PRIORIDADE']=='ALTA').sum()} | 2+ sinais: {(cand_f['N_MOTIVOS']>=2).sum()}")
cols = ['RANK_PRIORIDADE', 'FAIXA_PRIORIDADE', 'MOTIVOS', 'COD_EMPRESA', 'CODIGO', 'NIVEL_1',
        'RECEITA', 'MARGEM_BRUTA_PCT', 'DESCONTO_EFETIVO_PCT', 'DESVIO_VS_MEDIANA_PCT']
display(cand_f[cols].head(12).round(2))

Total candidatos (rede física): 3260 | ALTA: 326 | 2+ sinais: 225


,RANK_PRIORIDADE,FAIXA_PRIORIDADE,MOTIVOS,COD_EMPRESA,CODIGO,NIVEL_1,RECEITA,MARGEM_BRUTA_PCT,DESCONTO_EFETIVO_PCT,DESVIO_VS_MEDIANA_PCT
3489,1,ALTA,margem baixa + desconto alto + preco fora da f...,1,466115,I - ILUMINACAO,587.58,14.96,48.17,-45.15
3490,2,ALTA,margem baixa + desconto alto + preco fora da f...,92,454313,I - ILUMINACAO,388.29,17.91,32.64,-31.12
3491,3,ALTA,margem negativa + desconto alto + preco fora d...,7,388536,"G - PORTAS, JANELAS E FECHADURAS",335.79,-8.62,61.74,-60.79
3492,4,ALTA,desconto alto + preco fora da faixa,3,463933,D - ELETROS,106998.01,NaN,44.90,-33.36
3493,5,ALTA,desconto alto + preco fora da faixa,4,460689,O - MOVEIS,106421.07,NaN,57.89,-41.53
3494,6,ALTA,margem baixa + desconto alto,1,429455,E - MATERIAIS ELETRICOS,67624.59,17.56,62.14,-6.67
3495,7,ALTA,desconto alto + preco fora da faixa,3,427734,I - ILUMINACAO,47859.94,NaN,57.06,48.16
3496,8,ALTA,desconto alto + preco fora da faixa,92,481706,R - ELETRONICOS,41481.10,NaN,30.10,46.37
3497,9,ALTA,margem negativa + desconto alto,2,467615,B - UTILIDADES DOMESTICAS,27746.14,-29.58,33.11,-7.66
3498,10,ALTA,desconto alto + preco fora da faixa,4,451133,O - MOVEIS,24653.90,47.58,46.27,-30.40


## Revisão de qualidade (autoaudit antes/depois)

Revisão crítica dos próprios resultados: erros típicos de precificação/margem, a correção adotada e o impacto mensurável.

In [10]:
autoaudit = pd.read_csv(OUT / 'autoaudit_etapa5.csv', encoding='utf-8-sig')
for _, r in autoaudit.iterrows():
    display(Markdown(f"**{r['PROBLEMA']}**  \n*Correção:* {r['CORRECAO']}  \n*Antes:* {r['ANTES']}  \n*Depois:* {r['DEPOIS']}  \n*Impacto:* {r['IMPACTO']}"))

**Margens absurdas por erro de unidade (preco em caixa x custo)**  
*Correção:* Preco praticado = receita / qtd. em armazenagem; custo = preco de compra / conversao de compra. Tudo na mesma unidade.  
*Antes:* No nivel da linha, vendas em caixa (EMBALAGEM=1, conversao de venda ate 100) chegariam a dezenas de vezes o custo unitario se lidas por unidade de venda; 10 dos SKUs com custo vendem em caixa/conversao>1.  
*Depois:* Apos normalizar, os 261 SKUs com custo tem markup entre 0.65x e 4.76x e margem entre -52,8% e 79,0%.  
*Impacto:* 0 de 261 SKUs com markup acima de 20x (faixa sanitaria); a normalizacao funciona como salvaguarda contra o outlier de caixa.

**Custo de SKU sem compra vazando para a margem**  
*Correção:* Margem so e calculada para SKUs com custo proprio; os demais ficam sem margem, sinalizados como sem custo.  
*Antes:* 2468 SKUs (90,4%) e R$ 403,4M (83,6% da receita) poderiam receber margem fabricada.  
*Depois:* 261 SKUs com custo real entram na margem; 2468 ficam corretamente sem margem.  
*Impacto:* Margem realizada limitada a base auditavel, sem inflar rentabilidade.

**Mistura de embalagem (e atacado) na dispersao de preco**  
*Correção:* Dispersao por SKU x EMBALAGEM, com preco em unidade de armazenagem, dentro do universo (rede fisica separada do atacado).  
*Antes:* 2619 SKUs com amplitude de preco bruto >30% (todas as lojas, embalagens misturadas).  
*Depois:* 827 SKUs com amplitude >30% pela mesma metrica na rede fisica por embalagem; pelo CV, apenas 46 SKUs com CV>30%.  
*Impacto:* A maior parte da 'variacao de preco' era mistura de embalagem/atacado: cai de 2619 para 827 (46 pelo CV).

## Recomendações de melhoria

In [11]:
recomendacoes = pd.read_csv(OUT / 'recomendacoes_melhoria.csv', encoding='utf-8-sig')
display(recomendacoes[['PRIORIDADE', 'TEMA', 'LIMITACAO_OU_PROBLEMA', 'RECOMENDACAO']])

,PRIORIDADE,TEMA,LIMITACAO_OU_PROBLEMA,RECOMENDACAO
0,ALTA,Cobertura de custo,So ~261 SKUs (~16% da receita) tem preco de co...,Incorporar custo de reposicao/tabela de custo ...
1,ALTA,Camadas de custo (PEPS/medio movel),O custo usado e a media ponderada do periodo i...,Registrar custo por camada/lote (PEPS) e o cus...
2,MEDIA,Preco de lista x promocoes,O preco de lista (`dim_precos`) e um cadastro ...,Versionar o preco de lista com data de vigenci...
3,MEDIA,Embalagem e unidade,"Preco de venda, custo e lista vem em unidades ...",Padronizar um identificador de unidade de medi...
4,MEDIA,Canal atacado/B2B (Loja 93),"A Loja 93 opera como atacado/B2B, com preco e ...",Criar dimensao de canal e politicas de preco/m...


## Resumo interpretativo

In [12]:
display(Markdown((OUT / 'resumo_etapa5.md').read_text(encoding='utf-8')))

# Etapa 5 - Analise de precificacao e variacao de margem

## Glossario rapido (ler antes dos numeros)

- **Preco praticado (por unid. de armazenagem):** `receita / quantidade em
  unidade de armazenagem`. E o preco medio efetivamente realizado, ja na mesma
  unidade do custo.
- **Custo medio (CMV unitario):** media ponderada do preco de compra do SKU no
  periodo, convertido para a unidade de armazenagem
  (`preco de compra / conversao de compra`). So existe para SKUs com compra de
  preco valido.
- **Margem bruta (R$):** preco praticado - custo medio (mesma unidade).
- **Margem bruta (%):** margem R$ / preco praticado.
- **Markup:** preco praticado / custo medio (quantas vezes o custo o preco cobre).
- **Preco de lista:** preco de tabela cadastrado por loja x produto x embalagem.
- **Desconto efetivo (%):** `(preco de lista - preco praticado) / preco de lista`,
  sempre dentro da MESMA embalagem.
- **Dispersao de preco:** coeficiente de variacao do preco praticado do mesmo
  SKU entre lojas, por embalagem.
- **Repricing:** revisao de preco de itens com margem baixa/negativa, desconto
  fora do padrao ou preco fora da faixa da rede.

## Cobertura de custo (leia antes de generalizar a margem)

Margem realizada **so existe para os SKUs com custo de compra registrado**. Esse
e o analogo do achado das Etapas 1/2 ("88% vendem sem compra registrada"):

- 329 SKUs tem compra registrada no periodo, mas apenas
  261 tem preco de compra valido (9,5% das linhas de
  compra vem sem preco e sao excluidas do custo), de 2.729 SKUs vendidos.
- Esses SKUs respondem por R$ 79,1M
  (16,4% da receita) na rede completa e
  R$ 49,9M (15,2%)
  na rede fisica. Na Loja 93/B2B, a cobertura auditavel e
  R$ 19,7M (12,8%).
- Os demais SKUs **nao recebem margem por ausencia de dado, nunca por erro**.
  A margem deste relatorio vale para esse subconjunto, nao para a rede toda.

## Principais achados

- Margem bruta % media ponderada (apenas itens com custo): rede completa
  47,6% (markup 1.91x); rede
  fisica 49,8% (markup 1.99x).
- Categoria N1 de maior margem na rede fisica (cobertura de custo >=10%):
  `C - PISOS E REVESTIMENTOS` (56,1%, cobertura
  21,1%); de menor margem:
  `D - ELETROS` (42,8%, cobertura
  13,3%).
- Alerta: a categoria `B - UTILIDADES DOMESTICAS` aparece com margem NEGATIVA (-15,4%) no subconjunto com custo (cobertura de 2,7% da receita) - itens de baixo giro vendidos abaixo do custo, candidatos a repricing/descontinuacao.

- Entre os itens de alta receita (curva A) com custo na rede fisica, a menor
  margem e do SKU 447868 (PRATO RASO PORC.28CM    PEIXES 80000),
  com -52,8%; a maior e do SKU
  462811 (DISJ.BIPO.C DIN  16A       GII-C016A), com
  69,9%.
- 7 SKUs
  da rede fisica vendem com margem negativa (preco < custo), concentrados em
  utilidades domesticas/loucas de baixo giro - sinalizados, nao silenciados.
- Desconto efetivo medio ponderado na rede fisica: 17,4%.
  2.079 combinacoes loja x produto x embalagem vendem ACIMA da
  lista (desconto negativo), sinal de lista possivelmente desatualizada.
- Dispersao de preco entre lojas (rede fisica, embalagem 0): apenas
  46 SKUs com CV>30% - bem abaixo da leitura ingenua de
  2.619 SKUs com amplitude de preco bruto >30% (metrica diferente:
  amplitude do preco bruto entre linhas, misturando embalagem e atacado).
- Candidatos a repricing na rede fisica: 3.260 combinacoes
  loja x produto x embalagem com pelo menos um sinal (margem baixa/negativa,
  desconto alto ou preco fora da faixa). Dois rankings convivem: **risco**
  (nº de sinais + receita, para triagem) e **impacto** (receita x magnitude do
  sinal, para a fila comercial). O maior candidato por impacto na rede fisica e o
  SKU 480680 (REFRIG. 2P 391L FF RT38DG61  IX MBIV) na loja
  3, com R$ 1.723.994,28 de receita
  exposta (impacto estimado R$ 753.857,57),
  que no ranking de risco cai para o rank 226.

## Revisao de qualidade (autoaudit antes/depois)

- **Margens absurdas por erro de unidade (preco em caixa x custo).** No nivel da linha, vendas em caixa (EMBALAGEM=1, conversao de venda ate 100) chegariam a dezenas de vezes o custo unitario se lidas por unidade de venda; 10 dos SKUs com custo vendem em caixa/conversao>1. -> Apos normalizar, os 261 SKUs com custo tem markup entre 0.65x e 4.76x e margem entre -52,8% e 79,0%.
  (0 de 261 SKUs com markup acima de 20x (faixa sanitaria); a normalizacao funciona como salvaguarda contra o outlier de caixa.)
- **Custo de SKU sem compra vazando para a margem.** 2468 SKUs (90,4%) e R$ 403,4M (83,6% da receita) poderiam receber margem fabricada. -> 261 SKUs com custo real entram na margem; 2468 ficam corretamente sem margem.
- **Mistura de embalagem (e atacado) na dispersao de preco.** 2619 SKUs com amplitude de preco bruto >30% (todas as lojas, embalagens misturadas). -> 827 SKUs com amplitude >30% pela mesma metrica na rede fisica por embalagem; pelo CV, apenas 46 SKUs com CV>30%.
  (A maior parte da 'variacao de preco' era mistura de embalagem/atacado: cai de 2619 para 827 (46 pelo CV).)

## Limitacoes e cuidados

- Margem realizada existe so para os ~261 SKUs com custo
  (~16,4% da receita); o restante fica sem
  margem por ausencia de dado.
- O custo e a media ponderada do periodo, sem camadas de custo (PEPS) nem custo
  de reposicao atual.
- O preco de lista pode nao refletir promocoes pontuais; o desconto efetivo e uma
  aproximacao.
- A Loja 93 e atacado/B2B: margens nao comparaveis ao varejo, por isso segregada.
  A etapa gera um universo explicito `LOJA_93_ATACADO_B2B` para auditar esse canal.

## Validacoes

- 21 validacoes executadas, todas com status
  `OK`.
- Receita por universo, categoria e loja reconcilia com a Etapa 3, incluindo
  `LOJA_93_ATACADO_B2B`.
- Nenhuma margem % calculada sobre custo ausente; margens negativas sinalizadas.
- Preco praticado e de lista comparados apenas dentro da mesma embalagem.

## Como executar

```bash
.venv/Scripts/python.exe notebooks/etapa5_precificacao_margem.py
```

Os arquivos auditaveis sao gravados em `outputs/etapa5/`.
